In [12]:
import os
import numpy as np
import tensorflow as tf
import cv2
import glob
from tensorflow.keras.preprocessing.image import array_to_img, img_to_array

# Define augmentation parameters
def augment_image_and_mask(image, mask, data_gen_args):
    """
    Apply augmentations to image and mask.
    """
    image = array_to_img(image)
    mask = array_to_img(mask)

    # Convert to numpy arrays
    image = img_to_array(image)
    mask = img_to_array(mask)

    # Define ImageDataGenerator
    image_datagen = tf.keras.preprocessing.image.ImageDataGenerator(**data_gen_args)
    mask_datagen = tf.keras.preprocessing.image.ImageDataGenerator(**data_gen_args)

    # Create a batch of augmented images
    image_aug = next(image_datagen.flow(np.expand_dims(image, 0), batch_size=1))[0]
    mask_aug = next(mask_datagen.flow(np.expand_dims(mask, 0), batch_size=1))[0]

    # Ensure mask values are valid
    mask_aug = np.where(np.isin(mask_aug, [1, 2]), mask_aug, 0)

    return image_aug, mask_aug

# Directory paths
image_dir = 'enhanced_images/'
mask_dir = 'labelled_images/'

# Augmentation parameters
data_gen_args = dict(rotation_range=180,
                     width_shift_range=0.1,
                     height_shift_range=0.1,
                     shear_range=0.2,
                     zoom_range=0.2,
                     horizontal_flip=True,
                     vertical_flip=True,
                     fill_mode='nearest')

# Directories for output
augmented_images_dir = 'augmented_images/'
augmented_masks_dir = 'augmented_masks/'

# Create output directories if they don't exist
os.makedirs(augmented_images_dir, exist_ok=True)
os.makedirs(augmented_masks_dir, exist_ok=True)

# Iterate over all image and mask files
image_files = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.jpg")))

for img_path, mask_path in zip(image_files, mask_files):
    image = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    # Ensure the mask is resized appropriately
    mask = cv2.resize(mask, (128, 128), interpolation=cv2.INTER_NEAREST)
    
    augmented_image, augmented_mask = augment_image_and_mask(image, mask, data_gen_args)
    
    # Save augmented images and masks
    augmented_image = array_to_img(augmented_image)
    augmented_mask = array_to_img(augmented_mask.astype(np.uint8))  # Ensure mask is in uint8 format

    # Save images and masks
    base_name = os.path.basename(img_path).split('.')[0]
    augmented_image.save(os.path.join(augmented_images_dir, f'aug_{base_name}.png'))
    augmented_mask.save(os.path.join(augmented_masks_dir, f'aug_{base_name}.png'))

print("Augmentation and saving complete.")


ValueError: Expected image array to have rank 3 (single image). Got array with shape: (128, 128)